Imports and Environment Setup

## 1. Setup & Imports

In [ ]:
# Standard data & visualisation libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Google Colab
from google.colab import drive

# Scikit-learn (machine learning & preprocessing)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras (deep learning)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Set the overall visual style for the plots
sns.set_theme(style="whitegrid")

## 2. Load the Cleaned Dataset

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# 1. Load the cleaned data from T1.1
df = pd.read_csv('/content/drive/MyDrive/Machine Learning Project/cleaned_collision_data.csv')

# 2. Split into features (X) and target (y)
X = df.drop(columns=['collision_severity'])

# Remap target labels: 3=Slight -> 0, 2=Serious -> 1, 1=Fatal -> 2
label_mapping = {3: 0, 2: 1, 1: 2}
y = df['collision_severity'].map(label_mapping)

print("Data loaded and target remapped (0=Slight, 1=Serious, 2=Fatal).")
print(y.value_counts())

## 3. Baseline Model: Multinomial Logistic Regression
### Build Preprocessing Pipeline & Run Cross-Validation

In [ ]:
# 1. Define the feature lists to match the clean T1.1 dataset
numerical_features = [
    'longitude',
    'latitude',
    'speed_limit',
    'hour_of_day', # Added from the T1.1 cleanup
    'month'        # Added from the T1.1 cleanup
]

categorical_features = [
    'first_road_class',
    'road_type',
    'junction_detail',
    'second_road_class',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'urban_or_rural_area',
    'trunk_road_flag',
    'controlled_junction'
]

passthrough_features = ['is_weekend']

# 2. Build the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features),
        ('pass', 'passthrough', passthrough_features)
    ],
    remainder='drop'
)

# 3. Build the baseline pipeline
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

print("Pipeline successfully defined with the correct columns.\n")

# 4. Run cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Starting cross-validation (this may take a minute or two)...\n")

y_pred = cross_val_predict(baseline_pipeline, X, y, cv=skf, n_jobs=-1)

# 5. Output metrics
print("--- Baseline Logistic Regression Performance ---")
print(f"Overall Accuracy: {accuracy_score(y, y_pred):.4f}\n")
print("Detailed Classification Report:")
print(classification_report(
    y,
    y_pred,
    labels=[1, 2, 3],
    target_names=['1 (Fatal)', '2 (Serious)', '3 (Slight)']
))


### Sanity-Check Feature Columns

In [ ]:
# Sanity-check the feature column names
print(X.columns.tolist())


## 4. Relabel Target & Define a Shared Evaluation Function
### Majority-Class Baseline

In [ ]:
# 1. Safely restore y from the DataFrame and remap labels
# Old: 3=Slight, 2=Serious, 1=Fatal
# New: 0=Slight, 1=Serious, 2=Fatal
y = df['collision_severity'].copy()
label_mapping = {3: 0, 2: 1, 1: 2}

# Use .replace() only if original unmapped values (like 3) exist
if 3 in y.values:
    y = y.replace(label_mapping)

print("Labels remapped: 0=Slight, 1=Serious, 2=Fatal")
print(y.value_counts(dropna=False), "\n")

# 2. Universal evaluation function
# Every model is passed through this function to guarantee consistent reporting
def evaluate_model(pipeline, X, y, model_name):
    print(f"========== Evaluating {model_name} ==========")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Generate cross-validated predictions
    y_pred = cross_val_predict(pipeline, X, y, cv=skf, n_jobs=-1)

    # Calculate required metrics
    acc = accuracy_score(y, y_pred)
    bal_acc = balanced_accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    print(f"Overall Accuracy:  {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}\n")

    print("Classification Report:")
    print(classification_report(
        y,
        y_pred,
        labels=[0, 1, 2],
        target_names=['0 (Slight)', '1 (Serious)', '2 (Fatal)']
    ))

    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=['True Slight', 'True Serious', 'True Fatal'],
        columns=['Pred Slight', 'Pred Serious', 'Pred Fatal']
    ))
    print("========================================================\n")
    return y_pred

# Implement the majority-class baseline
majority_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='prior'))
])

# Evaluate the baseline and store predictions
majority_preds = evaluate_model(majority_pipeline, X, y, "Majority-Class Baseline")

### Re-Evaluate Logistic Regression (Relabelled Target)

In [ ]:
# Evaluate the multinomial logistic regression model
logreg_preds = evaluate_model(baseline_pipeline, X, y, "Multinomial Logistic Regression")

## 5. Random Forest Classifier

In [ ]:
# 1. Build the Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Re-using the same ColumnTransformer
    # Parameters:
    # n_estimators=100:      creates 100 distinct decision trees
    # max_depth=15:          prevents the trees from growing infinitely and overfitting the training data
    # class_weight='balanced': crucial for handling the minority Fatal class
    # n_jobs=-1:             uses all available CPU cores to speed up training
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Evaluate the Random Forest model
# Note: this may take 2-4 minutes to run because it trains 500 trees in total (100 trees x 5 folds)
rf_preds = evaluate_model(rf_pipeline, X, y, "Random Forest")


## 6. Deep Neural Network (DNN)

In [ ]:
print("========== Evaluating Deep Neural Network (DNN) ==========")
print("Starting 5-fold CV. This will take a few minutes...")

# 1. Define the K-fold strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Compute class weights for Keras (it requires a dictionary, unlike scikit-learn)
classes = np.unique(y)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weight_dict = {classes[i]: weights[i] for i in range(len(classes))}

# Arrays to store out-of-fold predictions and history
oof_preds = np.zeros(len(y))
fold_histories = []

# 3. Custom cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Training Fold {fold + 1}/5...")

    # Split the raw data
    X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Fit the preprocessor ONLY on the training data to prevent leakage, then transform both
    X_train_processed = preprocessor.fit_transform(X_train_raw)
    X_val_processed = preprocessor.transform(X_val_raw)

    # Scipy sparse matrices need to be dense arrays for TensorFlow
    if hasattr(X_train_processed, "toarray"):
        X_train_processed = X_train_processed.toarray()
        X_val_processed = X_val_processed.toarray()

    # 4. Build the DNN architecture
    model = Sequential([
        # Input layer matching the number of engineered features
        Dense(64, activation='relu', input_shape=(X_train_processed.shape[1],)),
        Dropout(0.3), # Regularisation to prevent overfitting
        Dense(32, activation='relu'),
        Dropout(0.3),
        # Output layer: 3 units for Slight/Serious/Fatal, using softmax for probabilities
        Dense(3, activation='softmax')
    ])

    # Compile with Adam optimiser and sparse categorical crossentropy (targets are 0, 1, 2)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Early stopping halts training if the validation loss stops improving, preventing overfitting
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # 5. Train the model and save the history
    history = model.fit(
        X_train_processed, y_train,
        validation_data=(X_val_processed, y_val),
        epochs=50,
        batch_size=256,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=0 # Keep the notebook output clean
    )

    fold_histories.append(history.history)

    # Predict the validation fold
    val_probs = model.predict(X_val_processed, verbose=0)
    oof_preds[val_idx] = np.argmax(val_probs, axis=1)

# 6. Final evaluation metrics
print("\n--- Final DNN Evaluation ---")
print(f"Overall Accuracy:  {accuracy_score(y, oof_preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y, oof_preds):.4f}\n")

print("Classification Report:")
print(classification_report(y, oof_preds, labels=[0, 1, 2], target_names=['0 (Slight)', '1 (Serious)', '2 (Fatal)']))

print("Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y, oof_preds),
    index=['True Slight', 'True Serious', 'True Fatal'],
    columns=['Pred Slight', 'Pred Serious', 'Pred Fatal']
))
print("========================================================\n")


### DNN Confusion Matrix

In [ ]:
# 1. Calculate the raw confusion matrix
cm = confusion_matrix(y, oof_preds)

# 2. Set up the visualisation
plt.figure(figsize=(8, 6))

# 3. Use Seaborn to create the heatmap
# annot=True puts the numbers in the boxes, fmt='d' ensures they are integers, cmap selects the colour theme
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred Slight', 'Pred Serious', 'Pred Fatal'],
    yticklabels=['True Slight', 'True Serious', 'True Fatal']
)

plt.title('DNN Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


### DNN Learning Curves

In [ ]:
# Select the first three folds to plot
folds_to_plot = [0, 1, 2]

# Width is set to 15 to comfortably fit 3 plots side-by-side
plt.figure(figsize=(15, 5))

for i, fold_idx in enumerate(folds_to_plot):
    history = fold_histories[fold_idx]

    # 1 row, 3 columns
    plt.subplot(1, 3, i + 1)
    plt.plot(history['loss'], label='Training Loss', color='blue', linewidth=2)
    plt.plot(history['val_loss'], label='Validation Loss', color='orange', linewidth=2)

    plt.title(f'DNN Learning Curve (Fold {fold_idx + 1})')
    plt.xlabel('Epochs')

    # Only add the Y-axis label to the far-left plot to keep it clean
    if i == 0:
        plt.ylabel('Loss (Sparse Categorical Crossentropy)')

    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 7. Model Comparison
### Balanced Accuracy vs. Fatal F1-Score

In [ ]:
# 1. Define the metrics from the evaluation outputs above
models = ['Majority Baseline', 'Logistic Regression', 'Random Forest', 'DNN']
balanced_accuracy = [0.3333, 0.4802, 0.4346, 0.4763]
fatal_f1_score = [0.00, 0.07, 0.09, 0.07]

# 2. Set up the bar chart
x = np.arange(len(models))
width = 0.35  # Width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, balanced_accuracy, width, label='Balanced Accuracy', color='#4C72B0')
rects2 = ax.bar(x + width/2, fatal_f1_score, width, label='Fatal F1-Score', color='#C44E52')

# 3. Add labels and formatting
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Balanced Accuracy vs. Fatal F1-Score')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 8. Feature Importance (Random Forest)

In [ ]:
print("Training Random Forest on the full dataset to extract feature importances...")
# 1. Fit the pipeline on the whole dataset
rf_pipeline.fit(X, y)

# 2. Extract the feature names from the ColumnTransformer
# get_feature_names_out() pulls the exact names of the one-hot encoded columns
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 3. Extract the importances from the Random Forest model
importances = rf_pipeline.named_steps['classifier'].feature_importances_

# 4. Create a DataFrame and sort it
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 5. Plot the top 15 most important features
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'].head(15)[::-1], importance_df['Importance'].head(15)[::-1], color='teal')
plt.xlabel('Relative Importance')
plt.title('Top 15 Most Predictive Features (Random Forest)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## 9. Confusion Matrix Comparison Across Models

In [ ]:
print("Generating confusion matrix comparison heatmaps...")

# Calculate confusion matrices using stored predictions
cm_logreg = confusion_matrix(y, logreg_preds)
cm_rf = confusion_matrix(y, rf_preds)
cm_dnn = confusion_matrix(y, oof_preds)

# Set up the 1x3 grid
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
labels = ['Slight', 'Serious', 'Fatal']

# Plot each heatmap
sns.heatmap(cm_logreg, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=labels, yticklabels=labels)
axes[0].set_title('Logistic Regression')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1], xticklabels=labels, yticklabels=labels)
axes[1].set_title('Random Forest')
axes[1].set_xlabel('Predicted Label')

sns.heatmap(cm_dnn, annot=True, fmt='d', cmap='Oranges', ax=axes[2], xticklabels=labels, yticklabels=labels)
axes[2].set_title('Deep Neural Network')
axes[2].set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrix Comparison Across Models', fontsize=16)
plt.tight_layout()
plt.show()

## 10. Precision-Recall Curve for the "Fatal" Class

In [ ]:
print("Extracting prediction probabilities... (this will take a moment)")

# 1. Extract the probabilities for Logistic Regression and Random Forest
logreg_probs = cross_val_predict(baseline_pipeline, X, y, cv=skf, n_jobs=-1, method='predict_proba')
rf_probs = cross_val_predict(rf_pipeline, X, y, cv=skf, n_jobs=-1, method='predict_proba')

# 2. Isolate the probabilities specifically for Class 2 (Fatal)
logreg_fatal_probs = logreg_probs[:, 2]
rf_fatal_probs = rf_probs[:, 2]

# Create a binary label array where Fatal is 1 and everything else is 0
y_fatal_binary = (y == 2).astype(int)

# 3. Calculate precision, recall, and average precision (AP) for each model
logreg_precision, logreg_recall, _ = precision_recall_curve(y_fatal_binary, logreg_fatal_probs)
logreg_ap = average_precision_score(y_fatal_binary, logreg_fatal_probs)

rf_precision, rf_recall, _ = precision_recall_curve(y_fatal_binary, rf_fatal_probs)
rf_ap = average_precision_score(y_fatal_binary, rf_fatal_probs)

# 4. Plot the PR curves
plt.figure(figsize=(10, 7))

plt.plot(logreg_recall, logreg_precision, label=f'Logistic Regression (AP = {logreg_ap:.3f})', color='blue', linewidth=2)
plt.plot(rf_recall, rf_precision, label=f'Random Forest (AP = {rf_ap:.3f})', color='green', linewidth=2)

# Plot a baseline representing random guessing (proportion of fatalities in the dataset)
baseline_ratio = y_fatal_binary.mean()
plt.axhline(y=baseline_ratio, color='red', linestyle='--', label=f'Random Guess Baseline ({baseline_ratio:.3f})')

plt.title('Precision-Recall Curve for the "Fatal" Class', fontsize=14)
plt.xlabel('Recall (True Positive Rate)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


### Precision-Recall Curve (Zoomed)

In [ ]:
# Plot the PR curves (Zoomed) using variables already stored in memory
plt.figure(figsize=(10, 7))

plt.plot(logreg_recall, logreg_precision, label=f'Logistic Regression (AP = {logreg_ap:.3f})', color='blue', linewidth=2)
plt.plot(rf_recall, rf_precision, label=f'Random Forest (AP = {rf_ap:.3f})', color='green', linewidth=2)

# Plot a baseline representing random guessing
plt.axhline(y=baseline_ratio, color='red', linestyle='--', label=f'Random Guess Baseline ({baseline_ratio:.3f})')

# --- Zoom in on the Y-axis to make the curve differences legible ---
plt.ylim(0, 0.25)

plt.title('Precision-Recall Curve for the "Fatal" Class (Zoomed to 0.25 Precision)', fontsize=14)
plt.xlabel('Recall (True Positive Rate)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 11. Final Performance Summary Table

In [ ]:
# Compile the final metrics gathered above into a dictionary
summary_data = {
    'Model': [
        'Majority-Class Baseline',
        'Multinomial Logistic Regression',
        'Random Forest',
        'Deep Neural Network (DNN)'
    ],
    'Overall Accuracy': [0.7516, 0.4793, 0.6388, 0.5472],
    'Balanced Accuracy': [0.3333, 0.4802, 0.4346, 0.4763],
    'Macro F1-Score': [0.2900, 0.3300, 0.3900, 0.3400],
    'Fatal F1-Score': [0.0000, 0.0700, 0.0900, 0.0700]
}

# Create the DataFrame
summary_table = pd.DataFrame(summary_data)

# Print the formatted table
print("Task 1.2: Model Performance Summary Table")
print("-" * 80)
print(summary_table.to_string(index=False, justify='center'))
print("-" * 80)


## 12. Export Dataset & Preprocessor

In [ ]:
# Package X, y, and the ColumnTransformer into a single file
export_bundle = {
    'X': X,
    'y': y,
    'preprocessor': preprocessor
}

# Save the bundle to the Machine Learning Project folder in Google Drive
joblib.dump(export_bundle, '/content/drive/MyDrive/Machine Learning Project/task1_2_export.joblib')

print("Dataset and preprocessor successfully exported to Google Drive!")


### Verify Saved Files

In [ ]:
# Confirm what has been saved in the current working directory
import os
print("Files saved here:", os.listdir())
